# Special Operators
MIMIQ offers further possibilities to create circuits, such as new gate declarations, or wrappers for common combinations of gates. 
The topics contains informations about:

- Gate declaration & Gate calls
- Block of instrucitons
- Repeated Operations
- Composite Gates
- pauli String
- Quantum Fourier Transform
- Phase Gradient
- Polynomial Oracle
- Diffusion
- More about composite gates
- Barrier

In [2]:
# modules
from mimiqcircuits import *
import numpy as np
from symengine import *

### Gate declaration & Gate calls

In [3]:
# Creation of an ansatz
rot = symbols("x")
@gatedecl("ansatz")
def ansatz(rot):
    c = Circuit()
    c.push(GateX(), 0)
    c.push(GateP(rot), 1)
    return c

c = ansatz(rot)

In [4]:
c = Circuit()
c.push(ansatz(np.pi/2), 0, 1)
c.draw()

        ┌────────┐                                                              
 q[0]: ╶┤0       ├─────────────────────────────────────────────────────────────╴
        │  ansatz│                                                              
 q[1]: ╶┤1       ├─────────────────────────────────────────────────────────────╴
        └────────┘                                                              
                                                                                


In [5]:
c = Circuit()
my_gate = ansatz(np.pi)
c.push(my_gate, 0, 1)
c.add_noise(my_gate, Depolarizing2(0.1)) 
c.draw()

        ┌────────┐  ┌───────────────────┐                                       
 q[0]: ╶┤0       ├──┤0                  ├──────────────────────────────────────╴
        │  ansatz│  │  Depolarizing(0.1)│                                       
 q[1]: ╶┤1       ├──┤1                  ├──────────────────────────────────────╴
        └────────┘  └───────────────────┘                                       
                                                                                


In [6]:
IfStatement(my_gate, BitString("111"))

IF (c==111) ansatz

### Blocks of Instruction

In [7]:
# From an existing circuit
circ = Circuit()
circ.push(GateH(), 0)
circ.push(GateCX(), 0, 1)

block = Block(circ)

#From a list of instructions
inst = [Instruction(GateX(), (0,)), Instruction(GateY(), ((1,)))]
block2 = Block(inst)

# Empty block with specified dimensions
block3 = Block(2,1,0) # 2qubit, 1 classical register, 0 z-variables


In [8]:
# Example Error correction code block
def error_detection_block():
    c = Circuit()
    c.push(GateCX(), 0, 1)
    c.push(GateCX(), 0, 2)
    c.push(MeasureZ(), 1, 0)
    c.push(MeasureZ(), 2, 1)
    c.push(IfStatement(GateX(), BitString("01")), 0, 0, 1) # Error correction
    c.push(IfStatement(GateX(), BitString("10")), 0, 0, 1) # Error correction
    return Block(c)

error_detection = error_detection_block()
error_detection

3-qubit, 2-bit block 123770d60 with 6 instructions:
├── CX @ q[0], q[1]
├── CX @ q[0], q[2]
├── M @ q[1], c[0]
├── M @ q[2], c[1]
├── IF (c==01) X @ q[0], c[0,1]
└── IF (c==10) X @ q[0], c[0,1]

In [9]:
# Add error detection block to a circuit
main_circuit = Circuit()
main_circuit.push(error_detection, 0, 1, 2, 0, 1)

main_circuit.push(GateH(), 1)

main_circuit.push(error_detection, 0, 1, 2, 0, 1)

main_circuit.draw()

        ┌──────────────────────┐   ┌──────────────────────┐                     
 q[0]: ╶┤0                     ├───┤0                     ├────────────────────╴
        │                      │┌─┐│                      │                     
 q[1]: ╶┤1  block 123770d60    ├┤H├┤1  block 123770d60    ├────────────────────╴
        │                      │└─┘│                      │                     
 q[2]: ╶┤2                     ├───┤2                     ├────────────────────╴
        └───────────╥──────────┘   └───────────╥──────────┘                     
                    ║                          ║                                
                    ║                          ║                                
 c:    ═════════════╩══════════════════════════╩════════════════════════════════
                   0,1                        0,1                               


### Working with blocks

In [10]:
# creation of an empty block with 2 qubits, 1 bits and 0 z-vars.
b = Block(2,1,0)
b

empty circuit

In [11]:
# Add instruction to the block
b.push(GateH(), 0)
b.push(GateX(), 1)
b.push(GateCX(), 0, 1)

2-qubit, 1-bit block 123770c30 with 3 instructions:
├── H @ q[0]
├── X @ q[1]
└── CX @ q[0], q[1]

In [12]:
# get the length of the circuit
len(b)

3

In [13]:
b[0]

H @ q[0]

In [14]:
# Trying to add operations that use more resources than the block dimensions will result in an error
try:
    b.push(GateZ(), 3)
except Exception as e:
    print(type(e).__name__)

ValueError


### Repeated Operations

There are two main ways to create repeated operations:
- Method 1: Using the Repeat constructor directly
Repeat(n, operations) # repeats the given operations 'n' times

- Method 2: Using the 'repeat' helper function
repeat(n, operations) # Repeats the operation with optional simplification logic

- Method 3: Using the '.repeat()' method on an operation instance
operation.repeat(n) # Shortband for repeating an operation 'n' times

In [15]:
# Example repeat X gate 3 times
Repeat(3, GateX())

∏³ X

In [16]:
# Example repeat RX gate 5 times
GateRX(Symbol("theta")).repeat(5)

∏⁵ RX(theta)

In [17]:
# Example repeat Hadamard gate 3 times
c = Circuit()
c.push(repeat(3, GateH()), 0)

1-qubit circuit with 1 instructions:
└── ∏³ H @ q[0]

In [18]:
# to see all the gates instead of block, use {decompose()} 
c.decompose()

1-qubit circuit with 3 instructions:
├── H @ q[0]
├── H @ q[0]
└── H @ q[0]

### Composite Gates
#### Pauli String
A PauliString is an $N$-qubit tensor product of Pauli operatords of the form:
$$P_1 \otimes P_2 \otimes P_3 \otimes ... \otimes P_N$$
where each $P_i \in \left\{ I,X,Y,Z  \right\} $ is a single qubit operator, including the identity

In [19]:
c = Circuit()
c.push(PauliString("IXYZ"), 1, 2, 3, 4)

5-qubit circuit with 1 instructions:
└── IXYZ @ q[1,2,3,4]

#### Quantum Fourier Transform
The QFT gate implements the The Quantum Fourier Transform which is a circuit used to realize a linear transformation on qubits and is a building block of many larger circuits such as Shor’s Algorithm or the Quantum Phase Estimation.

The QFT maps an arbitrary quantum state $|x\rangle = \sum_{j=0}^{N-1}x_j|j\rangle$ to a quantum state $\sum_{j=0}^{N-1}y_k|k\rangle$ according to the formula:

$$y_k = \frac{1}{\sqrt{N}} \sum_{j=0}^{N-1}x_jw_N^{-jk}$$

where $w_N = e^{2\pi i/N}$

In [20]:
# QFT for N = 5 qubits
c = Circuit()
c.push(QFT(5), 0, 1, 2, 3, 4)

5-qubit circuit with 1 instructions:
└── QFT @ q[0,1,2,3,4]

In [21]:
c.draw()

        ┌─────┐                                                                 
 q[0]: ╶┤0    ├────────────────────────────────────────────────────────────────╴
        │     │                                                                 
 q[1]: ╶┤1    ├────────────────────────────────────────────────────────────────╴
        │     │                                                                 
 q[2]: ╶┤2 QFT├────────────────────────────────────────────────────────────────╴
        │     │                                                                 
 q[3]: ╶┤3    ├────────────────────────────────────────────────────────────────╴
        │     │                                                                 
 q[4]: ╶┤4    ├────────────────────────────────────────────────────────────────╴
        └─────┘                                                                 
                                                                                


#### Phase Gradient

The PhaseGradient applies a phase shift to a quantum register of N qubits, where each computational basis state 
$|k\rangle$ experiences a phase proportional to its integer value k:

$$ \text{PhaseGradient} = \sum_{k=0}^{N-1}e^{i\frac{2\pi}{N}k}|k\rangle\langle k| $$

In [22]:
# Add phase gradient to 5 qubits
c = Circuit()
c.push(PhaseGradient(5), 0, 1, 2, 3, 4)

5-qubit circuit with 1 instructions:
└── PhaseGradient @ q[0,1,2,3,4]

In [23]:
c.draw()

        ┌───────────────┐                                                       
 q[0]: ╶┤0              ├──────────────────────────────────────────────────────╴
        │               │                                                       
 q[1]: ╶┤1              ├──────────────────────────────────────────────────────╴
        │               │                                                       
 q[2]: ╶┤2 PhaseGradient├──────────────────────────────────────────────────────╴
        │               │                                                       
 q[3]: ╶┤3              ├──────────────────────────────────────────────────────╴
        │               │                                                       
 q[4]: ╶┤4              ├──────────────────────────────────────────────────────╴
        └───────────────┘                                                       
                                                                                


#### Polynomial Oracle
Warning: The polynomialOracle works only with the state vector simulator and not with MPS, because of ancillas qubit use. 

The PolynomialOracle is a quantum oracle for a polynomial functonof two register. It applies a $\pi$ phase shift to any basis that satisfies $a.xy+b.x+c.y+d=0$, where $|x\rangle$ and $|y\rangle$ are the states of the two registers. 

In [24]:
c = Circuit()
c.push(PolynomialOracle(5,5,1,2,3,4), *range(10))

10-qubit circuit with 1 instructions:
└── PolynomialOracle(1, 2, 3, 4) @ q[0,1,2,3,4], q[5,6,7,8,9]

In [25]:
c.draw()

        ┌───────────────────────────────┐                                       
 q[0]: ╶┤0                              ├──────────────────────────────────────╴
        │                               │                                       
 q[1]: ╶┤1                              ├──────────────────────────────────────╴
        │                               │                                       
 q[2]: ╶┤2                              ├──────────────────────────────────────╴
        │                               │                                       
 q[3]: ╶┤3                              ├──────────────────────────────────────╴
        │                               │                                       
 q[4]: ╶┤4                              ├──────────────────────────────────────╴
        │   PolynomialOracle(1, 2, 3, 4)│                                       
 q[5]: ╶┤5                              ├──────────────────────────────────────╴
        │                   

#### Diffusion

The Diffusion operator corresponds to Grover's diffusion operator. It implements the unitary transformation:
$$H^{\otimes n}(1-2|0^n\rangle \langle0^n|)H^{\otimes n} $$

In [26]:
c  =Circuit()
c.push(Diffusion(10), *range(10))

10-qubit circuit with 1 instructions:
└── Diffusion @ q[0,1,2,3,4,5,6,7,8,9]

In [27]:
c.draw()

        ┌────────────┐                                                          
 q[0]: ╶┤0           ├─────────────────────────────────────────────────────────╴
        │            │                                                          
 q[1]: ╶┤1           ├─────────────────────────────────────────────────────────╴
        │            │                                                          
 q[2]: ╶┤2           ├─────────────────────────────────────────────────────────╴
        │            │                                                          
 q[3]: ╶┤3           ├─────────────────────────────────────────────────────────╴
        │            │                                                          
 q[4]: ╶┤4           ├─────────────────────────────────────────────────────────╴
        │   Diffusion│                                                          
 q[5]: ╶┤5           ├─────────────────────────────────────────────────────────╴
        │            │      

All composites gates can be decomposed using decompose() to extract their implementation

In [28]:
QFT(5).decompose()

5-qubit circuit with 15 instructions:
├── H @ q[4]
├── CP(0.5*pi) @ q[3], q[4]
├── H @ q[3]
├── CP(0.25*pi) @ q[2], q[4]
├── CP(0.5*pi) @ q[2], q[3]
├── H @ q[2]
├── CP(0.125*pi) @ q[1], q[4]
├── CP(0.25*pi) @ q[1], q[3]
├── CP(0.5*pi) @ q[1], q[2]
├── H @ q[1]
├── CP(0.0625*pi) @ q[0], q[4]
├── CP(0.125*pi) @ q[0], q[3]
├── CP(0.25*pi) @ q[0], q[2]
├── CP(0.5*pi) @ q[0], q[1]
└── H @ q[0]

In [29]:
QFT(5).decompose().draw()

                                                                                
 q[0]: ╶───────────────────────────────────────────────────────────────────────╴
                                                                                
 q[1]: ╶─────────────────────────────────────────────────●────────────●────────╴
                                                ┌─┐      │            │         
 q[2]: ╶───────────────────────●──────────●─────┤H├──────┼────────────┼────────╴
                      ┌─┐      │     ┌────┴────┐└─┘      │      ┌─────┴────┐    
 q[3]: ╶────────●─────┤H├──────┼─────┤P(0.5*pi)├─────────┼──────┤P(0.25*pi)├───╴
        ┌─┐┌────┴────┐└─┘┌─────┴────┐└─────────┘   ┌─────┴─────┐└──────────┘    
 q[4]: ╶┤H├┤P(0.5*pi)├───┤P(0.25*pi)├──────────────┤P(0.125*pi)├───────────────╴
        └─┘└─────────┘   └──────────┘              └───────────┘                
                                                                                
...
                        

#### Barrier
The Barrier is a non-op operation that does not affect the quantum state but prevents compression or optimization across execution. As of now, Barrier is only useful when combined with the MPS backend.

To add barriers to the circuit, you can use the Barrier operation:



In [30]:
c = Circuit()
c.push(GateX(), 0)

# apply barrier on 0
c.push(Barrier(1), 0)

# apply individual barriers on multiple qubits
c.push(GateX(), 0)
c.push(Barrier(1), range(3))

# apply one general barrier on multiple qubits
c.push(GateX(), range(3))
c.push(Barrier(3), *range(3))

3-qubit circuit with 10 instructions:
├── X @ q[0]
├── Barrier @ q[0]
├── X @ q[0]
├── Barrier @ q[0]
├── Barrier @ q[1]
├── Barrier @ q[2]
├── X @ q[0]
├── X @ q[1]
├── X @ q[2]
└── Barrier @ q[0,1,2]

In [31]:
c.draw()

        ┌─┐ ┌─┐   ┌─┐                                                           
 q[0]: ╶┤X├░┤X├░──┤X├──────░───────────────────────────────────────────────────╴
        └─┘░└─┘░  └─┘┌─┐   ░                                                    
 q[1]: ╶────────░────┤X├───░───────────────────────────────────────────────────╴
                ░    └─┘┌─┐░                                                    
 q[2]: ╶─────────░──────┤X├░───────────────────────────────────────────────────╴
                 ░      └─┘░                                                    
                                                                                
